# Logistic Regression with Normalized / Preprocessed Data

In [339]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
from skbio.stats.composition import clr

### Building Baseline Model

In [314]:
# read in data
zhou_abundances = pd.read_csv('./data/preprocessed/batch_corrected/normalized_zhou_genus.csv', index_col=0)
zhou_meta = pd.read_csv('./data/preprocessed/merged_dfs/zhou_genus_merged_df.csv', 
    usecols=['iMSMS_ID', 'treatment_status', 'disease'], index_col=0)

In [315]:
# take only the untreated patients for building model
untreated_idx = zhou_meta['treatment_status'] != 'Treated'
untreated_abundances = zhou_abundances[untreated_idx]
print(untreated_abundances.shape)

(785, 91)


In [316]:
# encode disease status as 0: Control and 1: MS
zhou_disease = pd.Series(zhou_meta.loc[untreated_idx, 'disease'])
zhou_disease = zhou_disease.replace({'Control': 0, 'MS': 1}).astype(int)
print(zhou_disease.value_counts())
zhou_disease.head()

disease
0    576
1    209
Name: count, dtype: int64


iMSMS_ID
71402-0001    0
71402-0009    0
71402-0017    0
71402-0022    0
71402-0027    0
Name: disease, dtype: int64

In [317]:
# train/test split data
XTrain, XTest, ytrain, ytest = train_test_split(untreated_abundances, zhou_disease, 
    test_size=0.2, random_state=42, stratify=zhou_disease)

In [318]:
# scale data
scaler = StandardScaler()
XSTrain = scaler.fit_transform(XTrain)
XSTest = scaler.transform(XTest)

In [319]:
# baseline model
base_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.5, class_weight='balanced').fit(XSTrain, ytrain)
predY = base_model.predict(XSTest)

accuracy_score(ytest, predY)

0.6496815286624203

### Try Grid Search to Get Optimal Hyperparameters for Baseline Model

In [344]:
lr_clf = LogisticRegression(class_weight='balanced').fit(XSTrain, ytrain)

# try grid search to test for different parameters
param_grid_lr = {
    "l1_ratio": np.linspace(0, 1.0, 10),
    "C": [0.001, 0.03, 0.1, 0.25, 0.5, 1, 3, 10],
    "penalty": ['elasticnet'],
    "solver": ['saga'],
    "max_iter": [2000]
}

# leave out 20% of data per fold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# do a grid search with the combinations of hyperparameters above
random_search = GridSearchCV(
    estimator=lr_clf,
    param_grid=param_grid_lr,
    cv=kfold,
    scoring='roc_auc'
)

# run the actual search
random_search.fit(XSTrain, ytrain)

print(f"Best ROC AUC: {random_search.best_score_:.3f}")
print("Best Hyperparameters:")
print(random_search.best_params_)

# get the best model
best_model = random_search.best_estimator_

predY = best_model.predict(XSTest)
print(f"Test accuracy of best model: {accuracy_score(ytest, predY)}")

Best ROC AUC: 0.685
Best Hyperparameters:
{'C': 0.1, 'l1_ratio': np.float64(1.0), 'max_iter': 2000, 'penalty': 'elasticnet', 'solver': 'saga'}
Test accuracy of best model: 0.6242038216560509


In [ ]:
# look at what the top microbiota are
all_coefs = best_model.coef_.flatten()
all_coefs = pd.Series(all_coefs, index=untreated_abundances.columns).sort_values(ascending=False, key=abs)

nonzero_coefs = all_coefs[all_coefs != 0]
print(f"Number of microbiota with a non-zero coefficient: {len(nonzero_coefs)}")
print(nonzero_coefs)
print(nonzer_coefs.index)

Number of microbiota with a non-zero coefficient: 35
coprococcus             -0.386775
gemella                 -0.222233
intestinimonas           0.191872
coprobacillus            0.184343
anaerostipes             0.172194
butyrivibrio             0.162137
lactococcus             -0.152714
paraprevotella          -0.146830
ruminococcus             0.124744
parabacteroides          0.114366
staphylococcus           0.104576
acidaminococcus         -0.098551
anaeromassilibacillus    0.096215
klebsiella               0.095640
odoribacter             -0.081613
lactobacillus            0.075064
faecalibacterium        -0.066637
peptococcus              0.065496
adlercreutzia           -0.059689
prevotella              -0.059601
eubacterium             -0.056428
haemophilus             -0.052533
actinomyces              0.050259
megasphaera             -0.047137
lachnoclostridium        0.039779
enterococcus            -0.037090
lawsonella               0.020788
oscillibacter            0.01

In [354]:
print(nonzero_coefs.index)

Index(['coprococcus', 'gemella', 'coprobacillus', 'anaerofustis',
       'tepidiphilus', 'draconibacterium', 'mageeibacillus',
       'paraburkholderia', 'acetanaerobacterium', 'delftia',
       'pseudopropionibacterium', 'flavobacterium', 'rhodobacter',
       'tepidanaerobacter', 'elizabethkingia', 'gordonibacter',
       'paraprevotella', 'eremococcus', 'butyrivibrio', 'sphaerochaeta',
       'lactococcus', 'morganella', 'lelliottia', 'beggiatoa',
       'staphylococcus', 'intestinimonas', 'labilibacter', 'acidaminococcus',
       'brachyspira', 'desulfosporosinus', 'cellulosilyticum', 'eubacterium',
       'hathewaya', 'alloiococcus', 'pectobacterium', 'odoribacter',
       'treponema', 'adlercreutzia', 'anaeromassilibacillus', 'rhizobium',
       'tatumella', 'gracilibacillus', 'basfia', 'mobilibacterium',
       'megasphaera', 'enterococcus', 'trichococcus', 'halomonas',
       'peptococcus', 'saccharicrinis', 'sporosarcina', 'murdochiella',
       'actinobaculum', 'phocaeicola',

### Try with the Entire Zhou Feature Set

Just an experiment to see if the entire feature set actually helps with accuracy.

In [ ]:
# read in the raw abundance data (not CLR-transformed yet)
zhou_genus = pd.read_csv('data/preprocessed/abundance_data/Cohort1_Zhou_Genus.csv', index_col=0)

In [ ]:
## CLR transform abundance data first

# need to add a pseudocount to deal with zeros
pseudo_count = 1e-6
pseudo_abundances = zhou_genus + pseudo_count

# apply the CLR transformation
clr_abundances = clr(pseudo_abundances)
clr_df = pd.DataFrame(clr_abundances, columns=zhou_genus.columns, index=zhou_genus.index)

# only take the untreated + control subjects
clr_untreated = clr_df[untreated_idx]
print(clr_untreated.shape)

clr_untreated.head()

(785, 546)


,cellvibrio,coprobacillus,flectobacillus,capnocytophaga,pseudoflavonifractor,beggiatoa,collinsella,candidatus marithrix,rhodobacter,cupriavidus,...,nitrosomonas,aquabacterium,roseateles,flavonifractor,bdellovibrio,herbaspirillum,oceanospirillum,selenomonas,christensenella,tetrasphaera
iMSMS_ID,,,,,,,,,,,,,,,,,,,,,
71402-0001,-1.698834,3.841386,-1.698834,1.077746,5.226121,-1.698834,6.146349,-1.698834,-1.698834,-1.698834,...,-1.698834,-1.698834,-1.035374,3.045415,-1.698834,-0.818274,-1.698834,0.195632,-0.043133,-1.698834
71402-0009,-1.791213,4.252284,-1.791213,0.518837,4.275097,-1.791213,8.283283,-1.791213,-1.510477,-1.510477,...,-1.791213,-1.791213,-1.791213,5.634096,-1.791213,-1.291524,-1.791213,0.486139,-1.791213,-1.791213
71402-0017,-1.395545,3.491638,-1.395545,-0.517053,3.772101,-1.395545,1.406599,-1.395545,-0.862782,-1.395545,...,-1.395545,-0.862782,-1.395545,7.812928,-1.395545,-1.395545,-1.395545,0.495919,-1.395545,0.112562
71402-0022,-1.646414,4.459884,-1.131679,0.697767,5.273407,-1.646414,6.000400,-1.646414,-1.646414,-1.646414,...,-1.646414,-0.793536,-1.646414,5.070588,-1.646414,-1.646414,-1.646414,1.307259,-0.340035,-1.646414
71402-0027,-1.760404,4.455475,-1.760404,0.432625,4.765274,-1.760404,8.705351,-1.760404,-1.760404,-1.760404,...,-1.760404,-1.760404,-1.760404,7.431759,-1.760404,-1.760404,-1.760404,1.214041,-1.760404,-1.760404


In [353]:
XTrain, XTest, ytrain, ytest = train_test_split(clr_untreated, zhou_disease, test_size=0.2, 
    random_state=42, stratify=zhou_disease)

scaler = StandardScaler()
XSTrain = scaler.fit_transform(XTrain)
XSTest = scaler.transform(XTest)

# run model
full_zhou_lr = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=2000, C=0.1,
                            l1_ratio=1, class_weight='balanced').fit(XSTrain, ytrain)
predY = full_zhou_lr.predict(XSTest)

accuracy_score(ytest, predY)

0.7197452229299363

In [346]:
# look at what the top microbiota are
all_coefs = full_zhou_lr.coef_.flatten()
all_coefs = pd.Series(all_coefs, index=clr_untreated.columns).sort_values(ascending=False, key=abs)

nonzero_coefs = all_coefs[all_coefs != 0]
print(f"Number of microbiota with a non-zero coefficient: {len(nonzero_coefs)}")

Number of microbiota with a non-zero coefficient: 96


### See how many of these 96 microbiota overlap with the 91 Filtered Features In-Common Between Datasets

In [350]:
features96 = nonzero_coefs.index

incommon = set(features96).intersection(set(zhou_abundances.columns))
print(f"{len(incommon)} top features from Zhou dataset alone that can also be found in Levi dataset")
print(incommon)

24 top features from Zhou dataset alone that can also be found in Levi dataset
{'intestinimonas', 'anaeromassilibacillus', 'enterococcus', 'lactobacillus', 'gemella', 'veillonella', 'porphyromonas', 'adlercreutzia', 'acidaminococcus', 'ruminococcus', 'coprococcus', 'eubacterium', 'butyrivibrio', 'prevotella', 'anaerostipes', 'staphylococcus', 'megasphaera', 'odoribacter', 'peptococcus', 'phascolarctobacterium', 'coprobacillus', 'lactococcus', 'paraprevotella', 'blautia'}
